In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# =======================
# APPLICATION LOG MLP (PRODUCTION STYLE)
# =======================

import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, metrics
import random, os, joblib, math
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, accuracy_score,
    precision_score, recall_score, f1_score,
    precision_recall_curve
)
from sklearn.utils.class_weight import compute_class_weight

# =======================
# SEED
# =======================
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# =======================
# LOAD DATA
# =======================
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/updated_dataset.csv.gz")
df.columns = df.columns.str.strip()

# =======================
# LABEL
# =======================
df["threat_label"] = df["threat_label"].replace("suspicious","malicious")
df["Label"] = df["threat_label"].map({"benign":0,"malicious":1})

# =======================
# FEATURE ENGINEERING
# =======================
def request_depth(path):
    return str(path).count("/")

def path_entropy(path):
    path = str(path)
    if len(path) == 0:
        return 0
    prob = [n/len(path) for n in Counter(path).values()]
    return -sum(p * math.log2(p) for p in prob)

df["timestamp"] = pd.to_datetime(df["timestamp"])
df["hour"] = df["timestamp"].dt.hour
df["is_night"] = ((df["hour"] < 6) | (df["hour"] > 22)).astype(int)

df["request_depth"] = df["request_path"].apply(request_depth)
df["path_entropy"] = df["request_path"].apply(path_entropy)
df["path_length"] = df["request_path"].apply(lambda x: len(str(x)))

# ratio-like features (similar philosophy to network logs)
df["depth_per_length"] = df["request_depth"] / (df["path_length"] + 1)
df["entropy_per_length"] = df["path_entropy"] / (df["path_length"] + 1)

df.fillna(0, inplace=True)

# =======================
# ENCODING
# =======================
encoders = {}
for col in ["protocol","action","log_type"]:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

joblib.dump(encoders, "/content/drive/MyDrive/Colab Notebooks/app_label_encoders.pkl")

# =======================
# FEATURES
# =======================
features = [
    "protocol","action","log_type",
    "bytes_transferred",
    "request_depth","path_entropy","path_length",
    "depth_per_length","entropy_per_length",
    "hour","is_night"
]

df = df[features + ["Label"]]

# =======================
# SPLIT
# =======================
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=SEED, stratify=df["Label"]
)

X_train = train_df[features]
X_test  = test_df[features]
y_train = train_df["Label"].values
y_test  = test_df["Label"].values

# =======================
# CLEAN
# =======================
def clean(X):
    return X.replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)

X_train = clean(X_train)
X_test  = clean(X_test)

# =======================
# SCALING (ROBUST)
# =======================
scaler = RobustScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_test  = scaler.transform(X_test).astype(np.float32)

X_train = np.clip(X_train, -10, 10)
X_test  = np.clip(X_test,  -10, 10)

joblib.dump(scaler, "/content/drive/MyDrive/Colab Notebooks/app_scaler.pkl")

# =======================
# CLASS WEIGHTS
# =======================
class_weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weight_dict = {0: 1.0, 1: 1.2}

# =======================
# MODEL
# =======================
l2 = keras.regularizers.l2(1e-4)

model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),

    layers.Dense(128, activation="relu", kernel_regularizer=l2),
    layers.BatchNormalization(),
    layers.Dropout(0.4),

    layers.Dense(64, activation="relu", kernel_regularizer=l2),
    layers.BatchNormalization(),
    layers.Dropout(0.4),

    layers.Dense(32, activation="relu", kernel_regularizer=l2),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(16, activation="relu", kernel_regularizer=l2),
    layers.Dropout(0.2),

    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.BinaryCrossentropy(),
    metrics=[
        metrics.Precision(name="precision"),
        metrics.Recall(name="recall"),
        metrics.AUC(name="roc_auc"),
        metrics.AUC(name="pr_auc", curve="PR")
    ]
)

# =======================
# CALLBACKS
# =======================
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_roc_auc", mode="max", patience=7, restore_best_weights=True
)

lr_scheduler = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6
)

# =======================
# TRAIN
# =======================
model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=4096,
    class_weight=class_weight_dict,
    callbacks=[early_stop, lr_scheduler],
    verbose=1
)

# =======================
# SAVE
# =======================
model.save("/content/drive/MyDrive/Colab Notebooks/app_mlp_model.keras")

# =======================
# THRESHOLD TUNING (PR CURVE)
# =======================
y_prob = model.predict(X_test).ravel()

precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)

best_idx = np.argmax(f1_scores * (recalls < 0.98))
best_threshold = thresholds[best_idx]

np.save("/content/drive/MyDrive/Colab Notebooks/app_threshold.npy", best_threshold)

# =======================
# FINAL EVALUATION
# =======================
y_pred = (y_prob > best_threshold).astype(int)

print("\n===== APPLICATION LOG MODEL =====")
print("Threshold:", best_threshold)
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC AUC  :", roc_auc_score(y_test, y_prob))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Epoch 1/20
227/227 ━━━━━━━━━━━━━━━━━━━━ 17s 48ms/step - loss: 0.1856 - pr_auc: 0.9741 - precision: 0.8889 - recall: 0.9162 - roc_auc: 0.9848 - val_loss: 0.3680 - val_pr_auc: 0.9916 - val_precision: 1.0000 - val_recall: 0.6359 - val_roc_auc: 0.9939 - learning_rate: 0.0010
Epoch 2/20
227/227 ━━━━━━━━━━━━━━━━━━━━ 8s 36ms/step - loss: 0.0525 - pr_auc: 0.9984 - precision: 0.9851 - recall: 0.9766 - roc_auc: 0.9991 - val_loss: 0.0401 - val_pr_auc: 0.9993 - val_precision: 1.0000 - val_recall: 0.9688 - val_roc_auc: 0.9996 - learning_rate: 0.0010
Epoch 3/20
227/227 ━━━━━━━━━━━━━━━━━━━━ 10s 45ms/step - loss: 0.0340 - pr_auc: 0.9994 - precision: 0.9961 - recall: 0.9864 - roc_auc: 0.9997 - val_loss: 0.0224 - val_pr_auc: 1.0000 - val_precision: 1.0000 - val_recall: 0.9890 - val_roc_auc: 1.0000 - learning_rate: 0.0010
Epoch 4/20
227/227 ━━━━━━━━━━━━━━━━━━━━ 18s 36ms/step - loss: 0.0242 - pr_auc: 0.9998 - precision: 0.9973 - recall: 0.9928 - roc_auc: 0.9999 - val_loss: 0.0148 - val_pr_auc: 1.0000 - va